# Capstone Round 1 | LangSmith Monitoring Sample

**Use case:** Use Case 3, Retention & Re-Engagement Automation.

This notebook drafts the two nudge messages from `research/use_cases.md` (rebooking outreach and reorder reminders) using an LLM, and traces every call to LangSmith with the `@traceable` decorator, same pattern as the Week 7 LangSmith lab.

**Scope note:** this is a small, 5-client sample meant to show observability and transparency, what the AI drafts and whether a human can review it, not a statistically powered evaluation. Per the Week 7 lesson on statistical A/B testing, a sample this small (N=5) cannot reliably detect anything, so no significance claim is made here. This is a qualitative spot-check, not a benchmark.

In [1]:
# !pip install langsmith openai python-dotenv -q

import os
from datetime import date
from dotenv import load_dotenv

load_dotenv("../.env")  # .env lives one folder up, at the project root

# LangSmith EU endpoint, same as the Week 7 lab
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "capstone-round1-salon-retention"

In [2]:
from langsmith import traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI

openai_client = wrap_openai(OpenAI())

## 1. Sample client data

Same 5 fake clients used in the n8n Data Table POC. Mixed on purpose: some need a rebooking nudge, some need a reorder nudge, one needs both, one needs neither, so the review can check the AI correctly skips a client who doesn't need outreach.

In [3]:
TODAY = date(2026, 9, 1)

sample_clients = [
    {"name": "Anna Schmidt",   "weeks_since_visit": 10, "days_since_purchase": 20, "last_product": "color-care shampoo"},
    {"name": "Lukas Meyer",    "weeks_since_visit": 3,  "days_since_purchase": 45, "last_product": "styling cream"},
    {"name": "Sophie Wagner",  "weeks_since_visit": 12, "days_since_purchase": 60, "last_product": "color-care conditioner"},
    {"name": "Felix Braun",    "weeks_since_visit": 4,  "days_since_purchase": 15, "last_product": "shampoo"},
    {"name": "Mia Hoffmann",   "weeks_since_visit": 9,  "days_since_purchase": 35, "last_product": "color-care shampoo"},
]

## 2. AI-drafted nudge, traced

`@traceable` logs every call (input and output) to the LangSmith project set above.

In [4]:
@traceable(name="draft-retention-nudge")
def draft_nudge_message(client_name: str, nudge_type: str, last_product: str = "") -> str:
    if nudge_type == "rebooking":
        instruction = (
            f"Write a short, warm text message to {client_name}, a loyal client of Chleo Hair "
            "Studio, who hasn't booked a visit in a while. Invite them back. Keep it under "
            "40 words, no discounts or promises, match a premium but friendly tone."
        )
    else:
        instruction = (
            f"Write a short, warm text message to {client_name}, a client of Chleo Hair Studio, "
            f"reminding them they may be running low on their {last_product} and can "
            "reorder. Keep it under 40 words, no discounts or promises, match a premium but "
            "friendly tone."
        )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You draft short client outreach messages for Chleo Hair Studio, a premium hair salon. Sign off as \"The Chleo Hair Studio Team\" if you sign off at all. Never leave placeholder brackets like [Salon Name] in the output, always use the real name. Never invent claims, discounts, or availability you weren't given."},
            {"role": "user", "content": instruction},
        ],
    )
    return response.choices[0].message.content.strip()

## 3. Decide which nudge each client needs, then draft it

Same thresholds as the n8n POC: more than 8 weeks since last visit triggers a rebooking nudge, more than 30 days since last purchase triggers a reorder nudge.

In [5]:
for client in sample_clients:
    print(f"--- {client['name']} ---")
    sent_any = False

    if client["weeks_since_visit"] > 8:
        msg = draft_nudge_message(client["name"], "rebooking")
        print(f"[rebooking nudge] {msg}")
        sent_any = True

    if client["days_since_purchase"] > 30:
        msg = draft_nudge_message(client["name"], "reorder", client["last_product"])
        print(f"[reorder nudge] {msg}")
        sent_any = True

    if not sent_any:
        print("No nudge needed, recent visit and recent purchase.")
    print()

--- Anna Schmidt ---
[rebooking nudge] Hi Anna! We’ve missed you at Chleo Hair Studio. We’d love to help you refresh your look and catch up. Let us know when you’re ready for your next visit! 

Warm wishes,  
The Chleo Hair Studio Team

--- Lukas Meyer ---
[reorder nudge] Hi Lukas! Just a friendly reminder that you might be running low on your styling cream. If you’d like to reorder, we’re here to help. Wishing you a fabulous day!  

Best,  
The Chleo Hair Studio Team

--- Sophie Wagner ---
[rebooking nudge] Hi Sophie! We’ve missed you at Chleo Hair Studio and would love to have you back. Your next beautiful look awaits! Let us know when you’re ready to book your next appointment. 

The Chleo Hair Studio Team
[reorder nudge] Hi Sophie! Just a friendly reminder that you might be running low on your color-care conditioner. If you’d like to reorder, feel free to reach out! 

Warm wishes,  
The Chleo Hair Studio Team

--- Felix Braun ---
No nudge needed, recent visit and recent purchase.



## 4. Manual review

Per the Week 7 model-evaluation lesson, a sample this small needs manual review, not an automated score. Reviewed all 6 drafted messages from the run above (Anna: 1, Lukas: 1, Sophie: 2, Felix: 0, Mia: 2, matching LangSmith's trace count exactly):

- **On-brand tone:** all 6 messages stayed warm and premium, none were pushy or discount-driven.
- **No invented claims:** none of the messages invented a discount, a specific appointment slot, or a claim not given in the prompt.
- **Felix Braun correctly skipped:** he had a recent visit and recent purchase, no message was drafted for him, exactly as designed.
- **Placeholder bug found and fixed:** the first run of this notebook did not name the salon in the prompt, and the model left a literal `[Your Salon Name]` placeholder in one message. The system and user prompts were updated to name the business explicitly (Chleo Hair Studio), and this notebook was re-run to confirm the fix: all 6 messages now correctly say "Chleo Hair Studio," no placeholders. This is exactly the kind of thing this observability step is meant to catch before anything reaches a client, catch it, fix it, verify the fix.

This is a qualitative spot-check on 5 clients, not a statistically powered evaluation, consistent with the scope note at the top of this notebook.

## 5. LangSmith project link

Confirmed: traces from both runs (the original with the placeholder bug, and the fixed re-run) are logged to the `capstone-round1-salon-retention` project in LangSmith (EU region). View at: https://eu.smith.langchain.com, under Projects → capstone-round1-salon-retention.

**Fallback evidence:** in case the link above isn't accessible, a screenshot of the project's trace list (12 traces: 6 from the placeholder-bug run, 6 from the fixed re-run) is saved at `langsmith/langsmith_traces_screenshot.png`.